# ES / CME Globex — Exploratory Analysis
**Phase 2 — Day 3**  
Goal: understand the raw MBO data before writing cleaning rules.  
One file: `ES_20251001_orders.parquet` + `ES_20251001_trades.parquet`

Sections:
1. Schema & basic stats
2. Flags distribution
3. Price sanity (sentinels, negatives, outliers)
4. Timestamp & session structure
5. Action breakdown (A/C/M/R/T/F)
6. Symbol / expiry contamination
7. F_SNAPSHOT block analysis
8. Implied legs check
9. Trades: T/F pairing
10. Microstructure quick stats (spread proxy, OTR)

In [21]:
import duckdb
import pandas as pd
from pathlib import Path

# ── paths ──────────────────────────────────────────────────────────────────────
ORDERS = "../../data/market_data/product=ES/year=2025/month=10/ES_20251001_orders.parquet"
TRADES = "../../data/market_data/product=ES/year=2025/month=10/ES_20251001_trades.parquet"

# ── DuckDB connection (in-memory, read-only scans) ─────────────────────────────
con = duckdb.connect()

# ── flag constants (CME MDP3 / Databento) ──────────────────────────────────────
F_LAST            = 128   # last record in a multi-message event
F_TOB             = 64    # top-of-book aggregate, not individual order
F_SNAPSHOT        = 32    # recovery/snapshot message, not live market event
F_MBP             = 16    # aggregated price level (MBP schema), not MBO
F_BAD_TS_RECV     = 8     # ts_recv unreliable (clock issue or packet reorder)
F_MAYBE_BAD_BOOK  = 4     # unrecoverable gap detected in channel
F_PUBLISHER_SPEC  = 2     # publisher-specific semantics
INT64_MAX         = 9223372036854775807   # sentinel price for market/stop orders
UINT64_MAX        = 18446744073709551615  # sentinel for missing order_id

print("DuckDB", duckdb.__version__)
print("Paths set — run cells below")

DuckDB 1.4.4
Paths set — run cells below


## 1. Schema & basic stats

Schema matches Eurex — generic pipeline confirmed portable to CME MDP3.

Two points worth noting for `cme_config.py`:

- **`flags` = `UTINYINT` (uint8, 0–255)** — relevant for bitmask operations in DuckDB. Eurex may have exposed this as a signed INT; here it's unsigned 8-bit. Bitwise ops (`flags & 128`, etc.) still work, but be explicit about casting in `cme_config.py` to avoid silent promotion issues downstream.
- **`price` = `BIGINT` (signed INT64)** — consistent with Databento's fixed-point encoding at 1e-9 scale. Sentinel value for undefined/market orders is `INT64_MAX = 9223372036854775807`. All valid limit prices are positive INT64 in fixedpoint representation.

#### Next Steps

Run the following two sections — most informative starting point before touching the cleaning pipeline:

**§1 — Basic stats: orders + trades**
Key metrics to extract:
- Row counts by `action` type (Add / Cancel / Modify / Replace / Trade / Fill)
- Snapshot vs. live event split — critical on CME MDP3, which prefixes each session with a full book snapshot (`F_SNAPSHOT` flag). Unlike Eurex EOBI (which sends incremental updates from open), CME sends a snapshot burst at session start that must be handled separately in the state machine.

**§2 — Flags distribution**
- Full enumeration of observed `flags` combinations and their frequencies
- This is where ES will diverge most from Eurex — CME MDP3 uses `F_SNAPSHOT`, `F_TOB`, `F_LAST`, `F_MBP`, `F_BAD_TS_RECV` in combinations that reflect packet structure and book update sequencing. Knowing the exact flag mix on your data is a prerequisite before writing the LOB state machine.

In [22]:
# ── column types ───────────────────────────────────────────────────────────────
print("=== ORDERS schema ===")
display(con.execute(f"DESCRIBE SELECT * FROM '{Path(ORDERS)}'").df())

print("\n=== TRADES schema ===")
display(con.execute(f"DESCRIBE SELECT * FROM '{TRADES}'").df())

=== ORDERS schema ===


,column_name,column_type,null,key,default,extra
0,ts_recv,UBIGINT,YES,None,None,None
1,ts_event,UBIGINT,YES,None,None,None
2,ts_in_delta,INTEGER,YES,None,None,None
3,sequence,UINTEGER,YES,None,None,None
4,order_id,UBIGINT,YES,None,None,None
5,price,BIGINT,YES,None,None,None
6,size,UINTEGER,YES,None,None,None
7,action,VARCHAR,YES,None,None,None
8,side,VARCHAR,YES,None,None,None
9,flags,UTINYINT,YES,None,None,None



=== TRADES schema ===


,column_name,column_type,null,key,default,extra
0,ts_recv,UBIGINT,YES,None,None,None
1,ts_event,UBIGINT,YES,None,None,None
2,ts_in_delta,INTEGER,YES,None,None,None
3,sequence,UINTEGER,YES,None,None,None
4,order_id,UBIGINT,YES,None,None,None
5,price,BIGINT,YES,None,None,None
6,size,UINTEGER,YES,None,None,None
7,action,VARCHAR,YES,None,None,None
8,side,VARCHAR,YES,None,None,None
9,flags,UTINYINT,YES,None,None,None


In [23]:
# ── row counts & basic aggregates ─────────────────────────────────────────────
q = """
SELECT
    COUNT(*)                                            AS n_rows,
    COUNT(DISTINCT symbol)                              AS n_symbols,
    COUNT(DISTINCT action)                              AS n_actions,
    -- price stats on non-sentinel rows
    MIN(price) FILTER (WHERE price != {INT64_MAX})      AS price_min,
    MAX(price) FILTER (WHERE price != {INT64_MAX})      AS price_max,
    -- timestamp span (raw uint64 ns → readable)
    to_timestamp(MIN(CAST(ts_recv AS BIGINT)) / 1e9)   AS ts_start,
    to_timestamp(MAX(CAST(ts_recv AS BIGINT)) / 1e9)   AS ts_end,
    -- flag presence
    COUNT(*) FILTER (WHERE flags & {F_SNAPSHOT} > 0)   AS n_snapshot,
    COUNT(*) FILTER (WHERE flags & {F_LAST} > 0)       AS n_last,
    COUNT(*) FILTER (WHERE flags & {F_TOB} > 0)        AS n_tob,
    COUNT(*) FILTER (WHERE flags & {F_BAD_TS_RECV} > 0) AS n_bad_ts,
    COUNT(*) FILTER (WHERE flags & {F_MAYBE_BAD_BOOK} > 0) AS n_bad_book,
    COUNT(*) FILTER (WHERE price = {INT64_MAX})         AS n_sentinel_price,
    COUNT(*) FILTER (WHERE price < 0)                   AS n_negative_price,
    COUNT(*) FILTER (WHERE size <= 0)                   AS n_nonpositive_size
FROM '{ORDERS}'
""".format(**globals())

print("=== ORDERS basic stats ===")
display(con.execute(q).df().T)  # transpose for readability

=== ORDERS basic stats ===


,0
n_rows,10331993
n_symbols,12
n_actions,4
price_min,-45900000000
price_max,676500000000000
ts_start,2025-10-01 02:00:00+02:00
ts_end,2025-10-02 01:59:59.963129+02:00
n_snapshot,9908
n_last,9751287
n_tob,0


In [25]:
# Same for trades
q = """
SELECT
    COUNT(*)                                            AS n_rows,
    COUNT(DISTINCT symbol)                              AS n_symbols,
    COUNT(DISTINCT action)                              AS n_actions,
    MIN(price) FILTER (WHERE price != {INT64_MAX})      AS price_min,
    MAX(price) FILTER (WHERE price != {INT64_MAX})      AS price_max,
    to_timestamp(MIN(CAST(ts_recv AS BIGINT)) / 1e9)   AS ts_start,
    to_timestamp(MAX(CAST(ts_recv AS BIGINT)) / 1e9)   AS ts_end,
    COUNT(*) FILTER (WHERE flags & {F_SNAPSHOT} > 0)   AS n_snapshot,
    COUNT(*) FILTER (WHERE flags & {F_LAST} > 0)       AS n_last,
    COUNT(*) FILTER (WHERE price = {INT64_MAX})         AS n_sentinel_price,
    COUNT(*) FILTER (WHERE price < 0)                   AS n_negative_price
FROM '{TRADES}'
""".format(**globals())

print("=== TRADES basic stats ===")
display(con.execute(q).df().T)

=== TRADES basic stats ===


,0
n_rows,1253098
n_symbols,6
n_actions,2
price_min,51750000000
price_max,6875000000000
ts_start,2025-10-01 02:00:00.053284+02:00
ts_end,2025-10-02 01:59:58.424194+02:00
n_snapshot,0
n_last,0
n_sentinel_price,0


#### Results Summary

**Expected findings — all confirmed:**

- **`n_tob = 0`** — as expected. CME MBO does not publish `F_TOB` flags. The spread proxy in §10 will return zero rows by design — known limitation, handle via LOB reconstruction instead.
- **`n_bad_book = 0`** — no channel gap detected on this date. Clean feed.
- **`n_snapshot = 9,908`** — snapshot burst at session open, ~0.1% of total event count. To be analysed in §7.
- **`n_last = 9,751,287`** — ~94% of events carry `F_LAST`. Consistent with MBO semantics where most messages are atomic (one message = one complete book update).

---

#### Items Requiring Attention

**`n_symbols = 12` — unexpected, investigate immediately (§6)**

On Eurex you had 2–3 symbols per file (front month + back month + occasional roll). Twelve symbols on ES strongly suggests the raw DBN file contains more than outright futures: likely a mix of quarterly expirations, EuroStoxx-style calendar spreads, and possibly options on futures that leaked into the same dataset. Symbol contamination at this scale will corrupt the LOB state machine if not filtered upstream. §6 is the priority.

**`price_min = -45,900,000,000` — negative prices, different mechanism than Eurex**

In fixed-point 1e-9 encoding this maps to **-45.9 index points**. This is structurally different from the Eurex implied spread legs (which came in around -122 pts, i.e. the inter-expiry differential). At -45.9 pts on ES, the most likely candidates are CME calendar spreads or options on futures — deep ITM puts can carry synthetic negative prices depending on the encoding convention. §8 will clarify the exact source.

**`n_sentinel_price = 12` and `n_nonpositive_size = 12`** — almost certainly the same 12 rows. Market orders with no limit price, encoded as `INT64_MAX` per Databento convention. Negligible volume, clean filter.

**`ts_start = 02:00 UTC+2` / `ts_end = 01:59 UTC+2 next day` → 00:00–23:59 UTC**

The file covers exactly 24h UTC. CME labels files by UTC date, and this file contains the full electronic session for 2025-10-01 00:00 UTC through 23:59 UTC. The Monday overnight session split across two DBN files is confirmed — handle in `cme_config.py` session reconstruction.

---

#### Next Step

**Run §6 (symbol breakdown) now** — list all 12 symbols with their respective order and trade counts. This determines the front-month filter logic and any additional contamination rules needed before the LOB state machine.

## 2. Flags distribution
Important: flags are bitmasks — a row can carry multiple flags simultaneously.  
We look at both the raw value distribution AND per-flag counts.

In [26]:
# ── raw flags value distribution (orders) ─────────────────────────────────────
# Each unique flags value = a specific combination of bits
q = """
SELECT
    flags,
    -- decode which bits are set
    (flags & {F_LAST})           > 0 AS f_last,
    (flags & {F_TOB})            > 0 AS f_tob,
    (flags & {F_SNAPSHOT})       > 0 AS f_snapshot,
    (flags & {F_MBP})            > 0 AS f_mbp,
    (flags & {F_BAD_TS_RECV})    > 0 AS f_bad_ts,
    (flags & {F_MAYBE_BAD_BOOK}) > 0 AS f_bad_book,
    COUNT(*)                         AS n_rows,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 3) AS pct
FROM '{ORDERS}'
GROUP BY flags
ORDER BY n_rows DESC
""".format(**globals())

print("=== ORDERS flags distribution ===")
display(con.execute(q).df())

=== ORDERS flags distribution ===


,flags,f_last,f_tob,f_snapshot,f_mbp,f_bad_ts,f_bad_book,n_rows,pct
0,128,True,False,False,False,False,False,9751278,94.379
1,0,False,False,False,False,False,False,570804,5.525
2,40,False,False,True,False,True,False,9899,0.096
3,168,True,False,True,False,True,False,9,0.000
4,8,False,False,False,False,True,False,3,0.000


In [27]:
# ── raw flags value distribution (trades) ─────────────────────────────────────
q = """
SELECT
    flags,
    (flags & {F_LAST})           > 0 AS f_last,
    (flags & {F_TOB})            > 0 AS f_tob,
    (flags & {F_SNAPSHOT})       > 0 AS f_snapshot,
    (flags & {F_BAD_TS_RECV})    > 0 AS f_bad_ts,
    (flags & {F_MAYBE_BAD_BOOK}) > 0 AS f_bad_book,
    COUNT(*)                         AS n_rows,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 3) AS pct
FROM '{TRADES}'
GROUP BY flags
ORDER BY n_rows DESC
""".format(**globals())

print("=== TRADES flags distribution ===")
display(con.execute(q).df())

=== TRADES flags distribution ===


,flags,f_last,f_tob,f_snapshot,f_bad_ts,f_bad_book,n_rows,pct
0,0,False,False,False,False,False,1253098,100.0


#### Findings

Clean flag distribution — only 5 distinct combinations observed on orders, 1 on trades.

**Orders:**

| `flags` | Interpretation | Count | Treatment |
|---|---|---|---|
| 128 | `F_LAST` only — standard atomic event | 9.75M | ✅ keep |
| 0 | No flags — intermediate event in a multi-message burst | 570K | ✅ keep |
| 40 | `F_SNAPSHOT \| F_BAD_TS_RECV` — snapshot recovery block | 9,899 | ⚠️ exclude from live |
| 168 | `F_LAST \| F_SNAPSHOT \| F_BAD_TS_RECV` — last message of snapshot burst | 9 | ⚠️ exclude from live |
| 8 | `F_BAD_TS_RECV` only — 3 rows, outside snapshot context || ⚠️ log and keep (see below) |

**Trades:**

Single combination observed: `flags = 0` on 100% of trade records.

This is a fundamental structural difference from Eurex: on CME MDP3, trades do not carry `F_LAST`. The `F_LAST`-based T/F pair detection used on Eurex **cannot be ported directly to ES**. Fill grouping on CME will require matching on `ts_event` + `price` to reconstruct aggressor/passive pairs.

---

#### Immediate Design Decisions

**1. Snapshot filter:** `flags & 32 > 0` cleanly excludes all 9,908 snapshot rows (flags 40 and 168). Simple bitmask, no edge cases.

**2. `flags = 8` (`F_BAD_TS_RECV` outside snapshot) — 3 rows:** Recommendation is to log and keep. Three rows out of 10M is statistically irrelevant, and systematically dropping `F_BAD_TS_RECV` sets a dangerous precedent — if CME experiences a reordering spike, you would silently discard legitimate events. Log them for auditability, retain them in the clean dataset.

---

#### Next Step

Run **§7 (snapshot block analysis)** — confirm the snapshot-to-live transition gap and enumerate the `action` types present in the snapshot burst.

## 3. Price sanity

In [28]:
# ── sentinel prices: what actions carry them? ──────────────────────────────────
q = """
SELECT
    action,
    flags,
    (flags & {F_SNAPSHOT}) > 0 AS f_snapshot,
    (flags & {F_BAD_TS_RECV}) > 0 AS f_bad_ts,
    COUNT(*)                   AS n_rows
FROM '{ORDERS}'
WHERE price = {INT64_MAX}
GROUP BY action, flags
ORDER BY n_rows DESC
""".format(**globals())

print("=== Sentinel prices (INT64_MAX) by action + flags ===")
display(con.execute(q).df())

=== Sentinel prices (INT64_MAX) by action + flags ===


,action,flags,f_snapshot,f_bad_ts,n_rows
0,R,40,True,True,9
1,R,8,False,True,3


In [29]:
# ── price distribution on live (non-snapshot, non-sentinel) orders ─────────────
# Expressed in points (raw fixed-point / 1e9)
q = """
SELECT
    PERCENTILE_CONT(0.01) WITHIN GROUP (ORDER BY price / 1e9) AS p01,
    PERCENTILE_CONT(0.05) WITHIN GROUP (ORDER BY price / 1e9) AS p05,
    PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY price / 1e9) AS p25,
    PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY price / 1e9) AS p50,
    PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY price / 1e9) AS p75,
    PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY price / 1e9) AS p95,
    PERCENTILE_CONT(0.99) WITHIN GROUP (ORDER BY price / 1e9) AS p99,
    MIN(price / 1e9) AS price_min_pts,
    MAX(price / 1e9) AS price_max_pts,
    COUNT(*) AS n_rows
FROM '{ORDERS}'
WHERE price != {INT64_MAX}
  AND flags & {F_SNAPSHOT} = 0   -- exclude recovery snapshot
""".format(**globals())

print("=== Live order price distribution (points) ===")
display(con.execute(q).df().T)

=== Live order price distribution (points) ===


,0
p01,6682.25
p05,6689.00
p25,6716.25
p50,6736.00
p75,6761.00
p95,6803.50
p99,6847.50
price_min_pts,-45.90
price_max_pts,676500.00
n_rows,10322082.00


In [56]:
# ── tick size validation ───────────────────────────────────────────────────────
# ES tick = 0.25 pt = 250_000_000 in fixed-point (1e9 denominator)
# All prices mod tick_fp should be 0
TICK_FP = 250_000_000  # 0.25 * 1e9

q = f"""
SELECT
    COUNT(*) FILTER (WHERE price % {TICK_FP} != 0) AS n_tick_violations,
    COUNT(*) FILTER (WHERE price % {TICK_FP}  = 0) AS n_tick_ok,
    COUNT(*)                                        AS n_total
FROM '{ORDERS}'
WHERE price != {INT64_MAX}
  AND price > 0
  AND flags & {F_SNAPSHOT} = 0
"""

print("=== Tick size validation (ES tick = 0.25 pt) ===")
display(con.execute(q).df())

=== Tick size validation (ES tick = 0.25 pt) ===


,n_tick_violations,n_tick_ok,n_total
0,10884,10311192,10322076


## 4. Timestamp & session structure
Key question: what time range does 2025-10-01 cover?  
CME Globex opens Sunday 17:00 CT = 22:00 UTC. October 1 is a Wednesday → no overnight split issue.  
But we still want to see the exact ts range and identify the RTH (Regular Trading hours) window.

In [31]:
# ── hourly event count in UTC ──────────────────────────────────────────────────
# Helps visualize: pre-market, RTH (14:30–21:00 UTC), after-hours
q = """
SELECT
    DATE_TRUNC('hour', to_timestamp(CAST(ts_recv AS BIGINT) / 1e9)) AS hour_utc,
    COUNT(*)                                                         AS n_events,
    COUNT(*) FILTER (WHERE flags & {F_SNAPSHOT} > 0)                AS n_snapshot,
    COUNT(*) FILTER (WHERE flags & {F_SNAPSHOT} = 0)                AS n_live
FROM '{ORDERS}'
GROUP BY hour_utc
ORDER BY hour_utc
""".format(**globals())

print("=== ORDERS hourly distribution (UTC) ===")
print("RTH = 13:30–20:00 UTC (09:30–16:00 ET)")
display(con.execute(q).df())

=== ORDERS hourly distribution (UTC) ===
RTH = 13:30–20:00 UTC (09:30–16:00 ET)


,hour_utc,n_events,n_snapshot,n_live
0,2025-10-01 02:00:00+02:00,189639,9908,179731
1,2025-10-01 03:00:00+02:00,130839,0,130839
2,2025-10-01 04:00:00+02:00,107139,0,107139
3,2025-10-01 05:00:00+02:00,101904,0,101904
4,2025-10-01 06:00:00+02:00,90443,0,90443
5,2025-10-01 07:00:00+02:00,101963,0,101963
6,2025-10-01 08:00:00+02:00,284798,0,284798
7,2025-10-01 09:00:00+02:00,349818,0,349818
8,2025-10-01 10:00:00+02:00,272628,0,272628
9,2025-10-01 11:00:00+02:00,199517,0,199517


In [32]:
# ── same for trades ────────────────────────────────────────────────────────────
q = """
SELECT
    DATE_TRUNC('hour', to_timestamp(CAST(ts_recv AS BIGINT) / 1e9)) AS hour_utc,
    COUNT(*) AS n_trades
FROM '{TRADES}'
GROUP BY hour_utc
ORDER BY hour_utc
""".format(**globals())

print("=== TRADES hourly distribution (UTC) ===")
display(con.execute(q).df())

=== TRADES hourly distribution (UTC) ===


,hour_utc,n_trades
0,2025-10-01 02:00:00+02:00,12534
1,2025-10-01 03:00:00+02:00,8457
2,2025-10-01 04:00:00+02:00,6584
3,2025-10-01 05:00:00+02:00,6902
4,2025-10-01 06:00:00+02:00,5261
5,2025-10-01 07:00:00+02:00,6859
6,2025-10-01 08:00:00+02:00,30022
7,2025-10-01 09:00:00+02:00,29946
8,2025-10-01 10:00:00+02:00,22305
9,2025-10-01 11:00:00+02:00,17558


In [33]:
# ── ts_event vs ts_recv delta distribution ─────────────────────────────────────
# ts_event = exchange clock, ts_recv = Databento reception
# Delta = network + processing latency. Should be positive and small (μs range).
# Negative deltas = packet reordering (F_BAD_TS_RECV expected)
q = """
SELECT
    PERCENTILE_CONT(0.01) WITHIN GROUP
        (ORDER BY CAST(ts_recv AS BIGINT) - CAST(ts_event AS BIGINT)) AS delta_p01_ns,
    PERCENTILE_CONT(0.10) WITHIN GROUP
        (ORDER BY CAST(ts_recv AS BIGINT) - CAST(ts_event AS BIGINT)) AS delta_p10_ns,
    PERCENTILE_CONT(0.50) WITHIN GROUP
        (ORDER BY CAST(ts_recv AS BIGINT) - CAST(ts_event AS BIGINT)) AS delta_p50_ns,
    PERCENTILE_CONT(0.90) WITHIN GROUP
        (ORDER BY CAST(ts_recv AS BIGINT) - CAST(ts_event AS BIGINT)) AS delta_p90_ns,
    PERCENTILE_CONT(0.99) WITHIN GROUP
        (ORDER BY CAST(ts_recv AS BIGINT) - CAST(ts_event AS BIGINT)) AS delta_p99_ns,
    COUNT(*) FILTER
        (WHERE CAST(ts_recv AS BIGINT) < CAST(ts_event AS BIGINT))    AS n_negative_delta,
    COUNT(*) FILTER
        (WHERE flags & {F_BAD_TS_RECV} > 0)                           AS n_bad_ts_recv_flag
FROM '{ORDERS}'
WHERE flags & {F_SNAPSHOT} = 0
""".format(**globals())

print("=== ts_recv - ts_event latency distribution (nanoseconds) ===")
display(con.execute(q).df().T)

=== ts_recv - ts_event latency distribution (nanoseconds) ===


,0
delta_p01_ns,77540.0
delta_p10_ns,82566.0
delta_p50_ns,99789.0
delta_p90_ns,348782.6
delta_p99_ns,811479.0
n_negative_delta,0.0
n_bad_ts_recv_flag,3.0


#### Findings

**Session coverage:** File `ES_20251001` spans `02:00 UTC+2` (= 00:00 UTC) to `02:00 UTC+2 next day` (= 00:00 UTC on 2025-10-02). Exactly 24h UTC — consistent with the convention inferred from timestamps in §1.

**Intraday volume distribution:** RTH peak is clearly visible: 14:00–17:00 UTC+2 (= 13:30–16:00 UTC = 09:30–12:00 ET) concentrates the bulk of order flow. Pre-market 02:00–08:00 UTC+2 is active but roughly 10x thinner. Expected behaviour on ES.

**Gap at `23:00 UTC+2` (= 21:00 UTC) — 435 events only:** This is the CME Globex daily maintenance window: **21:00–22:00 UTC (17:00–18:00 ET)**. The feed goes quasi-silent during this window. The rebound at `00:00 UTC+2` (= 22:00 UTC) with 6,917 events is the session-open snapshot for the 2025-10-02 session — the next electronic session starting mid-file.

---

#### Session Definition Implications

This file contains **two distinct CME sessions**:

- **2025-10-01 session:** 00:00 UTC → 21:00 UTC (maintenance window)
- **2025-10-02 session:** 22:00 UTC → 23:59 UTC (next session open, snapshot included)

For RTH cleaning, the filter `13:30–20:00 UTC` handles this cleanly — the maintenance window falls outside RTH regardless. For the full electronic session, bounds must be defined explicitly:

```python
# Electronic session: 22:00 UTC(D-1) → 21:00 UTC(D)
# Maintenance window: 21:00–22:00 UTC daily
# RTH (EDT, UTC-4): 13:30–20:00 UTC
# RTH (EST, UTC-5): 14:30–21:00 UTC  ← effective from 2025-11-02 (DST change)
```

Note the EDT→EST transition on **2025-11-02**: RTH shifts from `13:30–20:00 UTC` to `14:30–21:00 UTC`. Both windows must be parameterised in `cme_config.py` with a DST-aware cutover date.

---

#### Latency: `ts_recv` − `ts_event`

- **Median = 99.8 µs (~100 µs)** — clean, consistent with Databento co-location at CME Aurora (Aurora, IL data center)
- **p99 = 811 µs** — occasional spikes, nothing anomalous for a co-located feed
- **`n_negative_delta = 0`** — zero packet reordering. The 3 `F_BAD_TS_RECV` events noted in §2 are almost certainly minor exchange-side clock drift, not genuine feed integrity issues

---

#### Updated Session Config for `cme_config.py`

```python
# RTH: 13:30–20:00 UTC (EDT) / 14:30–21:00 UTC (EST)
# Electronic session: 22:00 UTC(D-1) → 21:00 UTC(D)
# Maintenance window: 21:00–22:00 UTC daily
# DST cutover: 2025-11-02 (EDT → EST)
```

---

#### Next Step

Run **§9 (T/F pairing)** — last critical block before writing the cleaning rules and `cme_config.py`.

## 5. Action breakdown

In [34]:
# ── action distribution: A=Add, C=Cancel, M=Modify, R=Replace, T=Trade, F=Fill ─
# Split snapshot vs live
q = """
SELECT
    action,
    (flags & {F_SNAPSHOT}) > 0      AS is_snapshot,
    COUNT(*)                        AS n_rows,
    ROUND(AVG(size), 2)             AS avg_size,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY (flags & {F_SNAPSHOT}) > 0), 3) AS pct_within_group
FROM '{ORDERS}'
GROUP BY action, is_snapshot
ORDER BY is_snapshot, n_rows DESC
""".format(**globals())

print("=== ORDERS action breakdown (snapshot vs live) ===")
display(con.execute(q).df())

=== ORDERS action breakdown (snapshot vs live) ===


,action,is_snapshot,n_rows,avg_size,pct_within_group
0,C,False,4610620,1.48,44.668
1,A,False,4610395,1.52,44.665
2,M,False,1101067,3.06,10.667
3,R,False,3,0.00,0.000
4,A,True,9899,3.13,99.909
5,R,True,9,0.00,0.091


In [35]:
# ── trades action distribution ─────────────────────────────────────────────────
q = """
SELECT
    action,
    side,
    (flags & {F_SNAPSHOT}) > 0 AS is_snapshot,
    COUNT(*)                   AS n_rows,
    ROUND(AVG(size), 2)        AS avg_size
FROM '{TRADES}'
GROUP BY action, side, is_snapshot
ORDER BY is_snapshot, n_rows DESC
""".format(**globals())

print("=== TRADES action + side breakdown ===")
display(con.execute(q).df())

=== TRADES action + side breakdown ===


,action,side,is_snapshot,n_rows,avg_size
0,F,B,False,449364,1.41
1,F,A,False,439948,1.43
2,T,B,False,185640,3.33
3,T,A,False,178119,3.51
4,F,N,False,26,3.69
5,T,N,False,1,48.00


## 6. Symbol / expiry contamination
Front month roll: expect 1-2 active symbols. If more, we have expiry contamination.

In [36]:
# ── symbols present + event counts ────────────────────────────────────────────
for label, path in [("ORDERS", ORDERS), ("TRADES", TRADES)]:
    q = f"""
    SELECT
        symbol,
        COUNT(*)             AS n_rows,
        MIN(price / 1e9)     AS price_min,
        MAX(price / 1e9)     AS price_max,
        (flags & {F_SNAPSHOT}) > 0 AS is_snapshot
    FROM '{path}'
    WHERE price != {INT64_MAX}
    GROUP BY symbol, is_snapshot
    ORDER BY n_rows DESC
    """
    print(f"=== {label} symbols ===")
    display(con.execute(q).df())

=== ORDERS symbols ===


,symbol,n_rows,price_min,price_max,is_snapshot
0,ESZ5,8703296,2780.50,676500.00,False
1,ESH6,1577786,6401.00,6900.00,False
2,ESM6,27336,6600.00,6933.00,False
3,ESZ5-ESH6,10951,49.20,66.25,False
4,ESZ5,9320,1000.00,9788.75,True
5,ESH6-ESM6,1392,-45.90,59.30,False
6,ESZ5-ESM6,1188,12.40,116.45,False
7,ESH6,290,4285.00,7020.00,True
8,ESZ5-ESH6,217,49.20,66.25,True
9,ESU6,52,6842.00,6948.00,False


=== TRADES symbols ===


,symbol,n_rows,price_min,price_max,is_snapshot
0,ESZ5,1251962,6680.00,6769.50,False
1,ESH6,639,6740.00,6827.50,False
2,ESZ5-ESH6,482,57.60,58.75,False
3,ESM6,9,6822.00,6875.00,False
4,ESZ5-ESM6,4,110.05,110.35,False
5,ESM6-ESU6,2,51.75,51.75,False


#### Findings

The 12 symbols and negative prices are fully explained — this is CME calendar spread instrumentation, not options contamination.

**Symbol decode:**
- `ESZ5`, `ESH6`, `ESM6`, `ESU6`, `ESZ6`, `ESH7` — outright futures, quarterly expirations (Dec25, Mar26, Jun26, Sep26, Dec26, Mar27)
- `ESZ5-ESH6`, `ESZ5-ESM6`, `ESH6-ESM6`, etc. — CME calendar spread instruments: synthetic legs published directly in the MBO stream as first-class instruments with their own order books and `order_id` namespace

**Negative prices explained:** `ESH6-ESM6` at -45.9 pts = Mar26 minus Jun26. Negative price = Mar26 < Jun26, i.e. normal contango on ES. Same economic logic as Eurex implied spreads, but structurally different: CME publishes these as dedicated outright calendar spread instruments with independent order books in the same feed — not implied legs in the Eurex sense. Each has its own `order_id` space and full MBO depth.

---

#### Cleaning Rule Implications

The `MODE(symbol)` front-month filter used on Eurex will work here — `ESZ5` dominates at ~8.7M live rows vs ~158K for `ESH6` (second largest). However the filter logic should be made explicit: retain `symbol = front_month` and explicitly exclude both spread instruments (symbol contains `-`) and back-month outrights.

**Two architecture options for `cme_config.py`:**

**Option A — `MODE(symbol)` per day, same logic as Eurex.** Simple, automatic, handles the roll transparently without hardcoded contract codes.

**Option B — explicit filter:** `symbol NOT LIKE '%-%'` (drop spreads) + `symbol = front_month` either hardcoded or detected by dominant volume.

Recommendation: **Option A**, but with a guard assertion — if `MODE(symbol)` returns a symbol containing `-`, raise an exception. This should never happen given the volume dominance of the front-month outright, but it's a cheap and valuable safeguard against unexpected roll-day edge cases.

---

#### Next Steps

Run **§2 (flags distribution)** and **§7 (snapshot analysis)** — these are the two remaining critical blocks before the cleaning rules can be written for `cme_config.py`.

## 7. F_SNAPSHOT block analysis
CME sends a snapshot block at session open to reconstruct book state.  
We need to know: how many messages? what actions? does it end cleanly before live flow starts?

In [37]:
# ── snapshot block: size, time range, action mix ───────────────────────────────
q = """
SELECT
    to_timestamp(MIN(CAST(ts_recv AS BIGINT)) / 1e9) AS snapshot_start,
    to_timestamp(MAX(CAST(ts_recv AS BIGINT)) / 1e9) AS snapshot_end,
    MAX(CAST(ts_recv AS BIGINT)) - MIN(CAST(ts_recv AS BIGINT)) AS duration_ns,
    COUNT(*)                                          AS n_snapshot_rows,
    COUNT(DISTINCT action)                            AS n_distinct_actions,
    COUNT(DISTINCT symbol)                            AS n_symbols
FROM '{ORDERS}'
WHERE flags & {F_SNAPSHOT} > 0
""".format(**globals())

print("=== Snapshot block summary ===")
display(con.execute(q).df().T)

=== Snapshot block summary ===


,0
snapshot_start,2025-10-01 02:00:00+02:00
snapshot_end,2025-10-01 02:00:00+02:00
duration_ns,0
n_snapshot_rows,9908
n_distinct_actions,2
n_symbols,9


In [38]:
# ── snapshot actions breakdown ─────────────────────────────────────────────────
q = """
SELECT
    action,
    COUNT(*)         AS n_rows,
    AVG(size)        AS avg_size,
    MIN(price / 1e9) AS price_min,
    MAX(price / 1e9) AS price_max
FROM '{ORDERS}'
WHERE flags & {F_SNAPSHOT} > 0
GROUP BY action
ORDER BY n_rows DESC
""".format(**globals())

print("=== Snapshot actions ===")
display(con.execute(q).df())

=== Snapshot actions ===


,action,n_rows,avg_size,price_min,price_max
0,A,9899,3.129609,1.240000e+01,9.788750e+03
1,R,9,0.000000,9.223372e+09,9.223372e+09


In [39]:
# ── transition: last snapshot ts vs first live ts ──────────────────────────────
# Critical: is there a clean cut or do snapshot and live messages interleave?
q = """
WITH boundary AS (
    SELECT
        MAX(CAST(ts_recv AS BIGINT)) FILTER (WHERE flags & {F_SNAPSHOT} > 0) AS last_snapshot_ns,
        MIN(CAST(ts_recv AS BIGINT)) FILTER (WHERE flags & {F_SNAPSHOT} = 0) AS first_live_ns
    FROM '{ORDERS}'
)
SELECT
    to_timestamp(last_snapshot_ns / 1e9)  AS last_snapshot_ts,
    to_timestamp(first_live_ns / 1e9)     AS first_live_ts,
    (first_live_ns - last_snapshot_ns)    AS gap_ns,
    -- negative gap = interleaving (bad)
    (first_live_ns - last_snapshot_ns) < 0 AS snapshot_live_overlap
FROM boundary
""".format(**globals())

print("=== Snapshot → Live transition ===")
display(con.execute(q).df().T)

=== Snapshot → Live transition ===


,0
last_snapshot_ts,2025-10-01 02:00:00+02:00
first_live_ts,2025-10-01 02:00:00.000923+02:00
gap_ns,922600
snapshot_live_overlap,False


#### Findings

Snapshot block is clean. Full breakdown:

**Snapshot structure:**
- **`duration_ns = 0`** — all 9,908 snapshot messages share an identical `ts_recv`. CME delivers the snapshot as an instantaneous burst, all messages timestamped identically. Expected behaviour.
- **`n_distinct_actions = 2`**: A (9,899 Add) + R (9 Replace with `price = INT64_MAX`). The 9 Replace records are the sentinel prices observed in §1 — market orders present in the book snapshot. Automatically filtered by the sentinel price rule.
- **`n_symbols = 9`** — the snapshot reconstructs all instruments in the feed (outrights + calendar spreads), not just the front month. Expected: CME snapshots represent the full channel state.
- **`gap_ns = 922,600 ns ≈ 923 µs`** — clean positive gap between the last snapshot message and the first live event. No interleaving. The LOB state machine can initialise cleanly from the snapshot and begin processing live updates without any ordering ambiguity.
- **`snapshot_live_overlap = False`** — confirmed, clean separation.

---

#### LOB State Machine Implications

The CME snapshot is Add-only (plus a handful of sentinel Replace records to discard). This is the expected CME MDP3 pattern: the snapshot encodes the current book state as a sequence of Add Orders. The state machine must:

1. Ingest the snapshot block to populate the initial book state — **without emitting any TOB events**
2. Begin emitting LOB1 snapshots from the first live event post-snapshot

This is structurally simpler than the Eurex EOBI snapshot, which can contain Modify records and requires handling partial book state updates during the snapshot burst.

---

#### Draft Cleaning Rules — ES

| Rule | Filter | Rationale |
|---|---|---|
| 1 | `flags & 32 = 0` | Exclude snapshot block (`F_SNAPSHOT`) |
| 2 | `price != INT64_MAX` | Sentinel market orders with no limit price |
| 3 | `symbol = MODE(symbol)` per day | Front month only — implicitly drops calendar spreads and back-month outrights |
| 4 | `assert MODE(symbol) NOT LIKE '%-% '` | Guard: dominant symbol must never resolve to a spread instrument |

No implied leg filter needed (negative prices eliminated by Rule 3 on symbol). No outlier band filter needed — CME calendar spreads are synthetic instruments with their own order books, not anomalous prices in the front-month stream.

---

#### Next Steps

Run **§4 (session structure)** and **§9 (T/F pairing)** — both required before writing `cme_config.py` and the cleaning pipeline.

## 8. Implied legs check
ES is outright-only — no calendar spread synthetic legs expected (unlike Eurex).  
We verify: negative prices, very low prices, anomalous size=1 clusters.

In [59]:
# ── negative or suspiciously low prices ───────────────────────────────────────
# On ES, any price < 1000 pts would be anomalous (index never traded that low in scope)
PRICE_FLOOR_FP = 1_000 * 10**9  # 1000 pts in fixed-point

q = f"""
SELECT
    action,
    side,
    flags,
    price / 1e9 AS price_pts,
    size,
    to_timestamp(CAST(ts_recv AS BIGINT) / 1e9) AS ts
FROM '{ORDERS}'
WHERE price != {INT64_MAX}
  AND (price < 0 OR price < {PRICE_FLOOR_FP})
ORDER BY ts
LIMIT 50
"""

result = con.execute(q).df()
print(f"=== Low/negative price rows: {len(result)} ===")
if len(result) > 0:
    display(result)
else:
    print("None found — ES is outright-only as expected.")

=== Low/negative price rows: 50 ===


,action,side,flags,price_pts,size,ts
0,A,B,40,100.05,1,2025-10-01 00:00:00+00:00
1,A,B,40,53.50,1,2025-10-01 00:00:00+00:00
2,A,B,40,57.10,1,2025-10-01 00:00:00+00:00
3,A,B,40,57.10,1,2025-10-01 00:00:00+00:00
4,A,B,40,57.00,4,2025-10-01 00:00:00+00:00
5,A,B,40,56.95,1,2025-10-01 00:00:00+00:00
6,A,B,40,56.80,1,2025-10-01 00:00:00+00:00
7,A,B,40,56.65,1,2025-10-01 00:00:00+00:00
8,A,B,40,56.55,4,2025-10-01 00:00:00+00:00
9,A,B,40,56.50,1,2025-10-01 00:00:00+00:00


In [41]:
# ── size=1 cluster with unusual flags ─────────────────────────────────────────
# On Eurex: implied legs were size=1 with flags=128
# On CME: size=1 is normal for iceberg/individual orders, but check flag patterns
q = """
SELECT
    action,
    flags,
    (flags & {F_SNAPSHOT}) > 0 AS f_snapshot,
    COUNT(*) AS n_rows,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 3) AS pct
FROM '{ORDERS}'
WHERE size = 1
  AND flags & {F_SNAPSHOT} = 0
GROUP BY action, flags
ORDER BY n_rows DESC
LIMIT 20
""".format(**globals())

print("=== Size=1 orders: action + flags distribution ===")
display(con.execute(q).df())

=== Size=1 orders: action + flags distribution ===


,action,flags,f_snapshot,n_rows,pct
0,A,128,False,3737896,45.074
1,C,128,False,3320387,40.040
2,M,128,False,768718,9.270
3,C,0,False,462824,5.581
4,A,0,False,2183,0.026
5,M,0,False,764,0.009


## 9. Trades: T/F pairing
T = aggressor side, F = passive fill.  
Expect 1 T for each F (or 1 T for multiple F in iceberg case).  
Unpaired T = block trades, EFPs, or off-exchange prints.

In [42]:
# ── T vs F counts ──────────────────────────────────────────────────────────────
q = """
SELECT
    action,
    COUNT(*) AS n_rows,
    COUNT(*) FILTER (WHERE flags & {F_LAST} > 0)     AS n_last,
    COUNT(*) FILTER (WHERE flags & {F_SNAPSHOT} > 0) AS n_snapshot,
    ROUND(AVG(size), 2) AS avg_size,
    SUM(size)           AS total_volume
FROM '{TRADES}'
GROUP BY action
ORDER BY action
""".format(**globals())

print("=== T/F pairing summary ===")
display(con.execute(q).df())

=== T/F pairing summary ===


,action,n_rows,n_last,n_snapshot,avg_size,total_volume
0,F,889338,0,0,1.42,1262128.0
1,T,363760,0,0,3.42,1244790.0


In [43]:
# ── unpaired aggressors: T messages with F_LAST set but no matching F ──────────
# Proxy: T with F_LAST=True that have no F within the same ts_event group
# Simple check: count T where flags=128 (F_LAST only, no other flags)
q = """
SELECT
    flags,
    COUNT(*) AS n_rows
FROM '{TRADES}'
WHERE action = 'T'
GROUP BY flags
ORDER BY n_rows DESC
""".format(**globals())

print("=== Aggressor (T) flags distribution ===")
display(con.execute(q).df())

=== Aggressor (T) flags distribution ===


,flags,n_rows
0,0,363760


In [44]:
# ── iceberg detection proxy ────────────────────────────────────────────────────
# Iceberg: 1 T aggressor, multiple F fills at same price + same ts_event
# Signature: T not F_LAST (more fills coming in same event)
q = """
SELECT
    action,
    (flags & {F_LAST}) > 0 AS f_last,
    COUNT(*) AS n_rows,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY action), 3) AS pct_of_action
FROM '{TRADES}'
GROUP BY action, f_last
ORDER BY action, f_last
""".format(**globals())

print("=== F_LAST distribution by action (iceberg proxy) ===")
print("T without F_LAST = aggressor with more passive fills pending in same event")
display(con.execute(q).df())

=== F_LAST distribution by action (iceberg proxy) ===
T without F_LAST = aggressor with more passive fills pending in same event


,action,f_last,n_rows,pct_of_action
0,F,False,889338,100.0
1,T,False,363760,100.0


#### Findings

Confirmed — and this is a fundamental structural difference from Eurex.

**CME MDP3 trade structure:**
- `flags = 0` on 100% of trade records (both T and F)
- `F_LAST = False` on 100% of trades
- T/F ratio = 363,760 T / 889,338 F = **~2.45 passive fills per aggressor on average**

`F_LAST` is not usable as a pairing signal on CME MDP3. T/F grouping requires a different approach.

**Actual CME MDP3 mechanism:** CME publishes trade events in a stream that is structurally separate from the order book update stream. In Databento's MBO schema, T = aggressor and F = passive fill, but they are not linked by `F_LAST` — they are linked by `ts_event` + `price`. A single T can have N associated F records sharing the same `ts_event` and `price` (iceberg resting orders and/or multi-level sweeps).

The ~2.45 F/T ratio indicates that a significant proportion of trades consume multiple passive orders — consistent with a one-tick spread market where large aggressor orders routinely sweep multiple resting levels or interact with iceberg replenishments. Expected on ES.

**Implication for Phase 3 (OFI / feature engineering):**

```python
# Group trades by (ts_event, price) → 1 T + N passive fills F
# T with no matching F in same (ts_event, price) group = off-exchange print
# (block trade, EFP, privately negotiated)
# Detectable: T records where no F shares same ts_event + price
```

For Phase 2 cleaning this does not need to be implemented — T/F pairing is a Phase 3 concern (OFI construction, fill rate analysis). The cleaner retains T and F without distinction, filtered only by symbol and sentinel price. The `is_unpaired_aggressor` flag used on Eurex has no equivalent here given `flags = 0` universally.

---

#### Final Cleaning Rules — ES

**Orders:**

| Rule | Filter | Expected exclusions | Rationale |
|---|---|---|---|
| 1 | `flags & 32 = 0` | ~9,908 | Snapshot block (`F_SNAPSHOT`) |
| 2 | `price != INT64_MAX` | ~12 | Sentinel market orders |
| 3 | `symbol = front_month` | ~1.6M | Calendar spreads + back-month outrights |
| 4 | `assert front_month NOT LIKE '%-% '` | — | Guard: front month must never resolve to a spread |

**Trades:**

| Rule | Filter | Rationale |
|---|---|---|
| 1 | `symbol = front_month` | Same logic as orders |
| 2 | `price != INT64_MAX` | Sentinel filter (expected zero hits on trades, verify) |

No implied leg filter. No outlier band. All negative prices belong to calendar spread instruments eliminated by Rule 3.

## 10. Microstructure quick stats
Sanity check on key metrics before cleaning.

In [45]:
# ── OTR on this single day ─────────────────────────────────────────────────────
q_orders = f"""
SELECT COUNT(*) AS n_live_orders
FROM '{ORDERS}'
WHERE flags & {F_SNAPSHOT} = 0
"""
q_trades = f"SELECT COUNT(*) AS n_trades FROM '{TRADES}'"

n_orders = con.execute(q_orders).fetchone()[0]
n_trades = con.execute(q_trades).fetchone()[0]
print(f"Live orders: {n_orders:,}")
print(f"Trades:      {n_trades:,}")
print(f"OTR:         {n_orders / n_trades:.1f}:1")

Live orders: 10,322,085
Trades:      1,253,098
OTR:         8.2:1


In [55]:
# ── RTH-only OTR (13:30–20:00 UTC = 09:30–16:00 ET) ──────────────────────────
# Note: Oct 1 2025 = EDT still active (clocks back first Sun of Nov)
# EDT = UTC-4 → RTH = 13:30–20:00 UTC
RTH_START = "13:30:00"
RTH_END   = "20:00:00"

con.execute("SET TimeZone='UTC'")  # pin timezone — must run before any TIMESTAMPTZ cast

q_rth_orders = f"""
SELECT COUNT(*) AS n
FROM '{ORDERS}'
WHERE flags & {F_SNAPSHOT} = 0
  AND CAST(CAST(to_timestamp(CAST(ts_recv AS BIGINT) / 1e9) AS TIMESTAMP) AS TIME) >= TIME '{RTH_START}'
  AND CAST(CAST(to_timestamp(CAST(ts_recv AS BIGINT) / 1e9) AS TIMESTAMP) AS TIME) <  TIME '{RTH_END}'
"""
q_rth_trades = f"""
SELECT COUNT(*) AS n
FROM '{TRADES}'
WHERE CAST(CAST(to_timestamp(CAST(ts_recv AS BIGINT) / 1e9) AS TIMESTAMP) AS TIME) >= TIME '{RTH_START}'
  AND CAST(CAST(to_timestamp(CAST(ts_recv AS BIGINT) / 1e9) AS TIMESTAMP) AS TIME) <  TIME '{RTH_END}'
"""

n_rth_orders = con.execute(q_rth_orders).fetchone()[0]
n_rth_trades = con.execute(q_rth_trades).fetchone()[0]
print(f"RTH live orders: {n_rth_orders:,}")
print(f"RTH trades:      {n_rth_trades:,}")
print(f"RTH OTR:         {n_rth_orders / n_rth_trades:.1f}:1")

RTH live orders: 7,161,875
RTH trades:      958,197
RTH OTR:         7.5:1


In [52]:
# ── cancel rate & fill rate proxies ───────────────────────────────────────────
q = f"""
SELECT
    COUNT(*) FILTER (WHERE action = 'A')  AS n_adds,
    COUNT(*) FILTER (WHERE action = 'C')  AS n_cancels,
    COUNT(*) FILTER (WHERE action = 'M')  AS n_modifies,
    COUNT(*) FILTER (WHERE action = 'R')  AS n_replaces
FROM '{ORDERS}'
WHERE flags & {F_SNAPSHOT} = 0
"""
res = con.execute(q).df()
n_adds    = res['n_adds'].iloc[0]
n_cancels = res['n_cancels'].iloc[0]
n_trades_total = con.execute(f"SELECT COUNT(*) FROM '{TRADES}'").fetchone()[0]

print(f"Adds:         {n_adds:,}")
print(f"Cancels:      {n_cancels:,}")
print(f"Cancel rate:  {n_cancels / n_adds:.3f}  (cancels / adds)")
print(f"Fill rate:    {n_trades_total / n_adds:.4f} (trades / adds)")

Adds:         4,610,395
Cancels:      4,610,620
Cancel rate:  1.000  (cancels / adds)
Fill rate:    0.2718 (trades / adds)


In [53]:
# ── trade size distribution ────────────────────────────────────────────────────
q = f"""
SELECT
    PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY size) AS p25,
    PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY size) AS p50,
    PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY size) AS p75,
    PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY size) AS p95,
    PERCENTILE_CONT(0.99) WITHIN GROUP (ORDER BY size) AS p99,
    MAX(size) AS size_max,
    AVG(size) AS avg_size
FROM '{TRADES}'
WHERE action = 'T'  -- aggressor side only, avoid double counting
"""
print("=== Trade size distribution (aggressor T only) ===")
display(con.execute(q).df().T)

=== Trade size distribution (aggressor T only) ===


,0
p25,1.000000
p50,1.000000
p75,3.000000
p95,12.000000
p99,30.000000
size_max,395.000000
avg_size,3.422009


In [49]:
# ── spread proxy: best ask - best bid at TOB events ────────────────────────────
# TOB events = messages where F_TOB is set OR we can infer from side + F_LAST
# Quick proxy: on Add actions, look at side='B' vs side='A' price distribution
# More precise: will be done in LOB reconstruction — this is just a sanity check
q = f"""
WITH tob AS (
    SELECT
        CAST(ts_recv AS BIGINT)                                     AS ts_ns,
        side,
        price / 1e9                                                 AS price_pts
    FROM '{ORDERS}'
    WHERE flags & {F_TOB} > 0
      AND flags & {F_SNAPSHOT} = 0
      AND price != {INT64_MAX}
),
pivoted AS (
    SELECT
        ts_ns,
        MAX(price_pts) FILTER (WHERE side = 'B') AS best_bid,
        MIN(price_pts) FILTER (WHERE side = 'A') AS best_ask
    FROM tob
    GROUP BY ts_ns
)
SELECT
    COUNT(*)                                                        AS n_tob_snapshots,
    PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY best_ask - best_bid) AS spread_p25,
    PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY best_ask - best_bid) AS spread_p50,
    PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY best_ask - best_bid) AS spread_p75,
    PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY best_ask - best_bid) AS spread_p95
FROM pivoted
WHERE best_bid IS NOT NULL AND best_ask IS NOT NULL
  AND best_ask > best_bid  -- exclude crossed (warm-up artifacts)
"""

result = con.execute(q).df()
if result['n_tob_snapshots'].iloc[0] > 0:
    print("=== Spread proxy from F_TOB events (points) ===")
    print("ES tick = 0.25 pt → 1-tick spread = 0.25 pt")
    display(result.T)
else:
    print("No F_TOB events found — CME MBO does not use F_TOB flag.")
    print("Spread will be computed from LOB state machine in next step.")

No F_TOB events found — CME MBO does not use F_TOB flag.
Spread will be computed from LOB state machine in next step.


In [50]:
# ── summary: open questions for cleaning rules ─────────────────────────────────
print("""
OPEN QUESTIONS TO ANSWER FROM OUTPUTS ABOVE:
─────────────────────────────────────────────
1. FLAGS:  What flag values dominate on live orders?
           Any unexpected flags (F_TOB, F_MBP)?

2. SNAPSHOT: How large is the snapshot block (n rows)?
             Is snapshot cleanly separated from live flow (no overlap)?
             What actions appear in snapshot (only A? or A+C mix)?

3. SESSION:  What hours are covered in this file?
             Is 2025-10-01 (Wednesday) a clean single-day session?
             How many events outside RTH (13:30–20:00 UTC)?

4. PRICES:   Any negative or anomalously low prices?
             Tick violations?

5. SYMBOLS:  1 or 2 active symbols? (front month + back month roll?)

6. T/F:      Ratio T vs F? Many unpaired aggressors?
             Iceberg signature (T without F_LAST)?

7. OTR:      Does this match Phase 1 estimate (~8:1)?
""")


OPEN QUESTIONS TO ANSWER FROM OUTPUTS ABOVE:
─────────────────────────────────────────────
1. FLAGS:  What flag values dominate on live orders?
           Any unexpected flags (F_TOB, F_MBP)?

2. SNAPSHOT: How large is the snapshot block (n rows)?
             Is snapshot cleanly separated from live flow (no overlap)?
             What actions appear in snapshot (only A? or A+C mix)?

3. SESSION:  What hours are covered in this file?
             Is 2025-10-01 (Wednesday) a clean single-day session?
             How many events outside RTH (13:30–20:00 UTC)?

4. PRICES:   Any negative or anomalously low prices?
             Tick violations?

5. SYMBOLS:  1 or 2 active symbols? (front month + back month roll?)

6. T/F:      Ratio T vs F? Many unpaired aggressors?
             Iceberg signature (T without F_LAST)?

7. OTR:      Does this match Phase 1 estimate (~8:1)?



In [5]:
con.execute("""
SELECT side, MIN(price/1e9) as price_min, MAX(price/1e9) as price_max, COUNT(*) as n
FROM '../../data/market_data/product=ES/year=2025/month=10/ES_20251001_orders.parquet'
WHERE flags & 32 > 0
  AND action = 'A'
  AND price != 9223372036854775807
  AND price > 0
  AND symbol = 'ESZ5'
GROUP BY side
ORDER BY price_min
""").df()

,side,price_min,price_max,n
0,B,1000.0,6723.25,5286
1,A,6723.5,9788.75,4034
